## 8. Feature Engineering — WOE Encoding & IV Ranking

In [ ]:
def calculate_woe_iv(dataframe, feature, target_col='default'):
    """Computes Weight of Evidence (WoE) and Information Value (IV) for a categorical column."""
    total_goods = (dataframe[target_col] == 0).sum()
    total_bads  = (dataframe[target_col] == 1).sum()

    df_counts = dataframe.groupby(feature)[target_col].agg(total_count='count', bads='sum')
    df_counts['goods'] = df_counts['total_count'] - df_counts['bads']
    df_counts['pct_goods'] = (df_counts['goods'] + 0.5) / total_goods  # epsilon avoids log(0)
    df_counts['pct_bads']  = (df_counts['bads']  + 0.5) / total_bads
    df_counts[f'{feature}_WoE'] = np.log(df_counts['pct_goods'] / df_counts['pct_bads'])
    df_counts['IV_component'] = (df_counts['pct_goods'] - df_counts['pct_bads']) * df_counts[f'{feature}_WoE']
    total_iv = df_counts['IV_component'].sum()

    print(f"--- {feature.upper()} (IV: {total_iv:.4f}) ---")
    print(df_counts[[f'{feature}_WoE', 'total_count']].round(4))
    print("-" * 50)

    return df_counts[f'{feature}_WoE'].to_dict(), total_iv


categorical_features = ['grade', 'purpose', 'home_ownership', 'emp_length']
iv_tracker = {}

for feature in categorical_features:
    if feature in df_cleaned.columns:
        woe_map, iv = calculate_woe_iv(df_cleaned, feature)
        iv_tracker[feature] = iv
        df_cleaned[f'{feature}_woe'] = df_cleaned[feature].map(woe_map)

print("\nCategorical WoE encoding complete.")

In [ ]:
def calculate_numerical_iv(dataframe, feature, target_col='default', bins=10):
    """Bins a numerical feature and computes its IV."""
    try:
        dataframe['binned'] = pd.qcut(dataframe[feature], q=bins, duplicates='drop')
        _, iv = calculate_woe_iv(dataframe, 'binned', target_col)
        dataframe.drop(columns=['binned'], inplace=True)
        return iv
    except Exception as e:
        print(f"Skipping {feature}: {e}")
        return None


numerical_features = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc'
]
for feature in numerical_features:
    if feature in df_cleaned.columns:
        iv = calculate_numerical_iv(df_cleaned, feature)
        if iv is not None:
            iv_tracker[feature] = iv

iv_summary = pd.DataFrame.from_dict(iv_tracker, orient='index', columns=['IV'])
iv_summary = iv_summary.sort_values('IV', ascending=False)
iv_summary['Predictive_Power'] = pd.cut(
    iv_summary['IV'],
    bins=[-np.inf, 0.02, 0.1, 0.3, 0.5, np.inf],
    labels=['Useless', 'Weak', 'Medium', 'Strong', 'Very Strong']
)
print(iv_summary)
print(f"\nFeatures to DROP (IV < 0.02): {iv_summary[iv_summary['IV'] < 0.02].index.tolist()}")

**IV Results:** `grade` is the single strongest categorical predictor (IV=0.46). `emp_length` is dropped (IV=0.002). `purpose` is borderline — keep WoE encoding but expect low model contribution.

**Final feature set:** `grade_woe`, `int_rate`, `fico_range_low`, `dti`, `mort_acc`, `loan_amnt`, `home_ownership_woe`, `annual_inc`, `revol_util`, `has_pub_rec`, `loan_to_income`

In [ ]:
print(f"Starting shape: {df_cleaned.shape}")

# Create binary flag before dropping raw pub_rec
if 'pub_rec' in df_cleaned.columns:
    df_cleaned['has_pub_rec'] = (df_cleaned['pub_rec'] > 0).astype(int)

explicit_drop_list = [
    'purpose', 'purpose_woe', 'open_acc', 'revol_bal',
    'emp_length', 'emp_length_woe', 'total_acc',
    'pub_rec',           # replaced by binary flag
    'fico_range_high',   # redundant with fico_range_low (r=1.0)
    'installment'        # redundant with loan_amnt (r=0.95)
]
columns_to_remove = [col for col in explicit_drop_list if col in df_cleaned.columns]
df_cleaned.drop(columns=columns_to_remove, inplace=True, errors='ignore')

print(f"Dropped: {columns_to_remove}")
print(f"Final shape after pruning: {df_cleaned.shape}")

In [ ]:
# 1. LOAN-TO-INCOME RATIO
df_cleaned['loan_to_income'] = df_cleaned['loan_amnt'] / df_cleaned['annual_inc'].clip(lower=1)

# 2. CREDIT UTILIZATION (fallback to revol_util if revol_bal was dropped)
if 'revol_bal' in df_cleaned.columns and 'total_rev_hi_lim' in df_cleaned.columns:
    df_cleaned['credit_util_calculated'] = (df_cleaned['revol_bal'] /
                                             df_cleaned['total_rev_hi_lim'].clip(lower=1))
else:
    print("→ Using baseline revol_util (revol_bal was dropped earlier).")

# 3. DEROGATORY COMPOSITE SCORE
delinq = df_cleaned['delinq_2yrs'].fillna(0)
pub_rec_flag = df_cleaned['has_pub_rec'].fillna(0)
pub_rec_bankruptcies = df_cleaned['pub_rec_bankruptcies'].fillna(0) if 'pub_rec_bankruptcies' in df_cleaned.columns else 0
df_cleaned['derog_composite'] = delinq + pub_rec_flag + pub_rec_bankruptcies

# 4. CREDIT AGE IN MONTHS
if 'earliest_cr_line' in df_cleaned.columns:
    df_cleaned['earliest_cr_line'] = pd.to_datetime(df_cleaned['earliest_cr_line'], format='mixed')
    df_cleaned['issue_d'] = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
    df_cleaned['credit_age_months'] = (
        (df_cleaned['issue_d'].dt.year - df_cleaned['earliest_cr_line'].dt.year) * 12 +
        (df_cleaned['issue_d'].dt.month - df_cleaned['earliest_cr_line'].dt.month)
    ).clip(lower=0)

print(f"Shape after feature engineering: {df_cleaned.shape}")

In [ ]:
for feat in ['loan_to_income', 'derog_composite', 'credit_age_months']:
    if feat in df_cleaned.columns:
        iv = calculate_numerical_iv(df_cleaned, feat)
        print(f"{feat}: IV = {iv:.4f}")

In [ ]:
# Drop derived features with low IV
df_cleaned.drop(columns=['derog_composite', 'credit_age_months'], inplace=True, errors='ignore')
print(f"Final modeling shape: {df_cleaned.shape}")

**Future work — LLM text feature extraction:** the `desc` column (free-text loan description, ~9% non-null) could be run through an LLM on a sample of loans to extract structured features such as a financial-stress score, purpose clarity, tone, or red flags — deferred here rather than implemented.